# Replication on Kaggle (unattended)

1. Settings: **Internet on**, **Accelerator: GPU T4 x2**.
2. **Add-ons -> Secrets**: add your Hugging Face token as `HF_TOKEN` and switch it on for this notebook (needed for Llama 3.1).
3. Set the switches in the next cell.
4. **Save Version -> Save & Run All (Commit)**. It runs without the browser; when it finishes, the files in `/kaggle/working` are kept in the version's **Output** tab.

Each stage resumes where it stopped if you run it again.

In [ ]:
RUN_MULTISALLM = False  # Qwen2.5-Coder-3B-Instruct (already done for Python/English)
RUN_CWEVAL = True       # Llama 3.1 8B Instruct
CWEVAL_CONFIGS = 't0.8p1 greedy'  # t0.8 = paper setting with model default top_p 0.9; t0.8p1 = top_p 1.0; greedy = Fig. 3
PLS = 'python'          # Multi-SALLM: 'python', 'python cpp', 'python cpp java'
LANGS = 'English'       # Multi-SALLM natural languages, comma separated, or 'all'
CWEVAL_N = 100          # samples per task at T=0.8 (paper: 100)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded:', os.environ['HF_TOKEN'][:3] + '...')
except Exception as e:
    print('No HF_TOKEN secret:', e)
os.environ.update(PLS=PLS, LANGS=LANGS, N=str(CWEVAL_N), CONFIGS=CWEVAL_CONFIGS)
%cd /kaggle/working
!test -d Graduation-Project || git clone -q --depth 1 -b claude/intelligent-noether-sgvujq https://github.com/leegundaniel/Graduation-Project.git
%cd Graduation-Project/replication
!git pull -q
!nvidia-smi -L

## Multi-SALLM

In [ ]:
if RUN_MULTISALLM:
    !./run_all.sh multisallm

In [ ]:
# (Multi-SALLM runs fully in the cell above)

## CWEval

In [ ]:
if RUN_CWEVAL:
    !./run_all.sh cweval
    !cat /kaggle/working/replication_work/results/cweval.md

## Keep the results
The small archive below holds the tables, logs and per-task test results, and appears in the version's Output tab.

In [ ]:
!cd /kaggle/working/replication_work && tar czf /kaggle/working/results_small.tgz results logs $(ls -d runs/cweval/*/res_all.json runs/cweval/*/pipeline.log runs/multisallm/*/Evaluation/Result 2>/dev/null); ls -lh /kaggle/working/results_small.tgz
!cp /kaggle/working/replication_work/results/*.md /kaggle/working/ 2>/dev/null; ls /kaggle/working